## 00. Imports

In [2]:
# ⚙️ Standard library
import json
import os
import shutil

# 🎨 Torchvision
import torchvision.transforms as T

# 🧱 Project modules
from src.helper import patches_from_coco
from roboflow import Roboflow

In [ ]:
DATASET_DIR = "dataset_project_iapr2025_temp"
VOC_DATASET_DIR = f"{DATASET_DIR}_voc"


API_KEY = "define your own"
#API_KEY="lLxKPsurNy6piDGR7GLc"

## 01. Download dataset in COCO format

Download annotated coco training dataset from roboflow

In [15]:
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("kaggleproject").project("kaggle_project")
version = project.version(1)
dataset = version.download("coco", location=DATASET_DIR)
# change name of train to train_annotated foolder
if os.path.exists(f"{DATASET_DIR}/train_annotated"):
    print("train_annotated folder already exists, skipping renaming.")
else:   
    shutil.move(f"{DATASET_DIR}/train", f"{DATASET_DIR}/train_annotated")


print(f"✅ Dataset downloaded to {DATASET_DIR}")

loading Roboflow workspace...
loading Roboflow project...
train_annotated folder already exists, skipping renaming.
✅ Dataset downloaded to dataset_project_iapr2025_temp


Download the training and testing datset from kaggle competition and merge it with the roboflow annotated version of training dataset

In [ ]:
#!kaggle competitions download -c chocolate-recognition-ml to 
#!kaggle competitions download -c chocolate-recognition-ml

 98%|█████████████████████████████████████ | 2.18G/2.24G [00:00<00:00, 3.15GB/s]
100%|██████████████████████████████████████| 2.24G/2.24G [00:00<00:00, 3.21GB/s]


In [24]:
# Source directory where you extracted the competition files
SOURCE_DIR = "chocolate-recognition-ml"
!kaggle competitions download -c chocolate-recognition-ml to 
!unzip -q chocolate-recognition-ml.zip -d {SOURCE_DIR}

# Make sure the dataset directory exists
os.makedirs(DATASET_DIR, exist_ok=True)

# Copy folders
for folder in [f"dataset_project_iapr2025/test", 
               f"dataset_project_iapr2025/train", 
               f"dataset_project_iapr2025/references"]:
    src = os.path.join(SOURCE_DIR, folder)
    folder = folder.replace(f"dataset_project_iapr2025/", "")
    dst = os.path.join(DATASET_DIR, folder)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

# Copy files
for file_name in ["dataset_project_iapr2025/train.csv",
                   "dataset_project_iapr2025/sample_submission.csv"]:
    src = os.path.join(SOURCE_DIR, file_name)
    file_name = file_name.replace("dataset_project_iapr2025/", "")
    dst = os.path.join(DATASET_DIR, file_name)
    shutil.copy(src, dst)

print(f"✅ Copied test/, train/, references/, train.csv, and sample_submission.csv to '{DATASET_DIR}'")

403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/competitions/data/download-all/to
✅ Copied test/, train/, references/, train.csv, and sample_submission.csv to 'dataset_project_iapr2025_temp'


In [ ]:
# Delete SOURCE_DIR
shutil.rmtree(SOURCE_DIR)
print(f"✅ Deleted '{SOURCE_DIR}' directory")

# Delete zip
if os.path.exists("chocolate-recognition-ml.zip"):
    os.remove("chocolate-recognition-ml.zip")
    print(f"✅ Deleted 'chocolate-recognition-ml.zip' file")

✅ Deleted 'chocolate-recognition-ml.zip' file


Patch training dataset, using annotations from coco format

In [51]:
# Cell 2: Define paths and load annotation info
train_img_dir = f"{DATASET_DIR}/train_annotated"
ann_path = os.path.join(train_img_dir, "_annotations.coco.json")

# Load COCO annotations and count classes
with open(ann_path, "r") as f:
    coco_json = json.load(f)
categories = coco_json["categories"]
num_classes = len(categories) + 1  # +1 for background
print("Detected classes:", [c["name"] for c in categories])
print("Total (with background):", num_classes)

# Run it on your dataset
source_path = f"{DATASET_DIR}/train_annotated"
destination_path = f"{DATASET_DIR}/train_patches"
patches_from_coco(source_path, destination_path)

Detected classes: ['objects', 'Amandina', 'Arabia', 'Comtesse', 'Creme_brulee', 'Jelly_Black', 'Jelly_Milk', 'Jelly_White', 'Noblesse', 'Noir_authentique', 'Passion_au_lait', 'Stracciatella', 'Tentation_noir', 'Triangolo']
Total (with background): 15


100%|██████████| 584/584 [00:33<00:00, 17.56it/s]

✅ Saved 584 patches and labels to dataset_project_iapr2025_temp/train_patches/patches


To patch the test dataset, you must first run the trained SSDLiteMobileNetV3 model to generate predictions.

•	This step is implemented in **section 02.e** of the train_models.ipynb notebook.

•	It identifies chocolate locations using the trained object detector before patch extraction.

👉 Check the notebook here:
**📄 train_models.ipynb**

In [54]:
# Check if test pacthes exist
test_patches_dir = f"{DATASET_DIR}/test_patches"
if not os.path.exists(test_patches_dir):
    print(f"❌ Test patches not found")
    print(f"Run train_models.ipynb to generate test patches")

❌ Test patches not found
Run train_models.ipynb to generate test patches


## 02. Download dataset in VOC format

In [15]:
project = rf.workspace("kaggleproject").project("kaggle_project")
version = project.version(1)
dataset = version.download("voc", location=VOC_DATASET_DIR)

# VOC dataset ready to work with add some nice print
print(f"VOC dataset downloaded to {VOC_DATASET_DIR}")
print("Ready to be used!")

loading Roboflow workspace...
loading Roboflow project...
VOC dataset downloaded to dataset_project_iapr2025_temp_voc
Ready to be used!
